In [1]:
import string
import nltk
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
nltk.download('stopwords')
from nltk.corpus import stopwords
nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Anandhu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
punc = string.punctuation

In [4]:
df = pd.read_csv("emotion_sentimen_dataset.csv")
df

,Unnamed: 0,text,Emotion
0,0,i seriously hate one subject to death but now ...,hate
1,1,im so full of life i feel appalled,neutral
2,2,i sit here to write i start to dig out my feel...,neutral
3,3,ive been really angry with r and i feel like a...,anger
4,4,i feel suspicious if there is no one outside l...,neutral
...,...,...,...
839550,839550,i feel like telling these horny devils to find...,neutral
839551,839551,i began to realize that when i was feeling agi...,neutral
839552,839552,i feel very curious be why previous early dawn...,neutral
839553,839553,i feel that becuase of the tyranical nature of...,neutral


In [5]:
df.isna().sum()
df.duplicated().sum()
df = df.dropna()
df = df.drop_duplicates()
df.isna().sum()
df.duplicated().sum()

df['Emotion'].value_counts()

Emotion
neutral       674538
love           39553
happiness      27175
sadness        17481
relief         16729
hate           15267
anger          12336
fun            10075
enthusiasm      9304
surprise        6954
empty           5542
worry           4475
boredom          126
Name: count, dtype: int64

In [6]:
love_count = len(df[df['Emotion'] == 'love'])
neutral_sample = df[df['Emotion'] == 'neutral'].sample(love_count, random_state=10)
love_rows = df[df['Emotion'] == 'love']
other_rows = df[~df['Emotion'].isin(['neutral', 'love'])]

df = pd.concat([love_rows, neutral_sample, other_rows])
df.reset_index(drop=True, inplace=True)

df['Emotion'].value_counts()

Emotion
love          39553
neutral       39553
happiness     27175
sadness       17481
relief        16729
hate          15267
anger         12336
fun           10075
enthusiasm     9304
surprise       6954
empty          5542
worry          4475
boredom         126
Name: count, dtype: int64

In [7]:
df.columns
df.drop(columns = "Unnamed: 0",inplace = True)
df.columns

Index(['text', 'Emotion'], dtype='object')

In [8]:
df['text'] = df['text'].astype(str)
stop_words = set(stopwords.words('english'))


In [9]:
def preprocess_text(text):
    text = "".join([i for i in text if i not in punc])
    text = text.lower()

    # Tokenize with SpaCy
    doc = nlp(text)

    # Remove stopwords and lemmatize
    tokens = [j.lemma_ for j in doc if j.text not in stop_words and not j.is_space]
    return " ".join(tokens)

df['Cleaned_text'] = df['text'].apply(preprocess_text)

count_vec = CountVectorizer()
count_vec.fit_transform(df['Cleaned_text'])

<204570x33957 sparse matrix of type '<class 'numpy.int64'>'
	with 2016300 stored elements in Compressed Sparse Row format>

j.text:        
Refers to the raw text of the token j. For example, if the token is the word "running," j.text would return "running."

stop_words:         
A set (or list) of common words that are typically filtered out in NLP tasks, like "the," "and," "is," etc. These words don't carry much meaning by themselves for many NLP tasks, 
so they're often removed.

j.lemma_:            
This refers to the lemma of the token j. Lemmatization is the process of reducing a word to its base or dictionary form. For example:
"running" → "run"
"better" → "good"
"geese" → "goose"
The lemma_ attribute returns the lemmatized form of the word (a string), whereas lemma returns a Token object.

j.is_space:               
Checks if the token is a space (i.e., a whitespace character like a space or tab). The condition not j.is_space ensures that spaces are excluded from the final list of tokens.

if j.text not in stop_words and not j.is_space:             
Filters out tokens that are either stop words (i.e., j.text is in the stop_words list) or spaces (j.is_space is True).

In [10]:
df['Cleaned_text']

0         feel jealous becasue want kind love true conne...
1         feel like take role grandmother since beloved ...
2         feel like back arm beloved last see long time ago
3         feel festive right lovely wintry scene walk do...
4                       discuss even feeling beloved anyone
                                ...                        
204565    feel like muscle around eye something funny ge...
204566    little pity detest feel pity go pain unfortuna...
204567    feel weepy sad ask unbelievably kind patient h...
204568    encounter lately include feeling like universe...
204569      feel miss error fear rejection failure whatever
Name: Cleaned_text, Length: 204570, dtype: object

In [11]:
from sklearn.model_selection import train_test_split

X = df['Cleaned_text']
y = df['Emotion'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_count_vec = count_vec.fit_transform(X_train)
X_test_count_vec = count_vec.transform(X_test)

X_train_count_vec


<163656x32384 sparse matrix of type '<class 'numpy.int64'>'
	with 1613101 stored elements in Compressed Sparse Row format>

In [12]:
# from sklearn.preprocessing import MaxAbsScaler

# scaler = MaxAbsScaler()
# X_train_count_vec = scaler.fit_transform(X_train_count_vec)
# X_test_count_vec = scaler.transform(X_test_count_vec)

In [13]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_count_vec,y_train)

score = model.score(X_test_count_vec,y_test)
score

0.905949063890111

In [14]:
import joblib

joblib.dump(model, 'Emotion_analysis.joblib')
joblib.dump(count_vec, 'CountVectorizer_Emotion_analysis.joblib')


['CountVectorizer_Emotion_analysis.joblib']

In [15]:
test_df = pd.concat((X_test,y_test),axis=1)
test_df.to_csv("TESTED_X.csv")

CHECKING WITH TF-IDF

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['Cleaned_text'])

In [17]:
X = df['Cleaned_text']
y = df['Emotion'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

X_train_tfidf


<163656x32384 sparse matrix of type '<class 'numpy.float64'>'
	with 1613101 stored elements in Compressed Sparse Row format>

In [18]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_tfidf,y_train)

score = model.score(X_test_tfidf,y_test)
score

0.6737058219680305